# Occupation-Only Sentence-BERT Winner Model

This notebook compares each contestant's `occupation_text_for_model` with the category and clue text on each Jeopardy! board. It computes preregistered similarity and within-game advantage features, then fits a regularized multinomial logistic regression.

High-value clues are ordinary J!/DJ! clues whose value is at least 0.8 of that game's maximum value in the same round. Final Jeopardy is not included in that feature.

Only board-visible text and `occupation_text_for_model` are predictors. Scores, wagers, answers, winner IDs, names, historical winnings, and postgame metadata are excluded. The train/test split is a reproducible 80/20 split by whole game; baseline fitting is deferred.

In [5]:
from pathlib import Path
import html
import re

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd().resolve().parent
DATA_DIR = PROJECT_DIR / "Data"
OUTPUT_DIR = PROJECT_DIR / "Output"
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 20260925
TEST_SIZE = 0.20
HIGH_VALUE_RATIO = 0.80
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
ARTIFACT_PREFIX = "occupation_only_model"

FINAL_PATH = DATA_DIR / "final_contestant_games.csv"
RAW_CLUE_PATH = DATA_DIR / "jeopardy_raw_contestant_clue_data.csv.gz"

final_columns = [
    "game_id", "contestant_id", "contestant_position", "is_winner",
    "occupation_text_for_model",
]
raw_columns = [
    "game_id", "clue_source_record_id", "round", "category",
    "clue_text", "clue_value",
]

contestants = pd.read_csv(
    FINAL_PATH,
    skipinitialspace=True,
    usecols=lambda column: column.strip() in final_columns,
)
contestants.columns = contestants.columns.str.strip()
contestants = contestants[final_columns].copy()
raw_clues = pd.read_csv(
    RAW_CLUE_PATH,
    compression="gzip",
    skipinitialspace=True,
    usecols=lambda column: column.strip() in raw_columns,
)
raw_clues.columns = raw_clues.columns.str.strip()
raw_clues = raw_clues[raw_columns].copy()
raw_clues = raw_clues[raw_clues["game_id"].isin(contestants["game_id"].unique())].copy()

print(f"Contestant-game rows: {len(contestants):,}")
print(f"Games: {contestants['game_id'].nunique():,}")
print(f"Raw clue rows after game filter: {len(raw_clues):,}")

Contestant-game rows: 11,646
Games: 3,882
Raw clue rows after game filter: 676,164


In [6]:
def clean_text(value):
    value = "" if pd.isna(value) else str(value)
    value = html.unescape(re.sub(r"<[^>]+>", " ", value))
    return re.sub(r"\s+", " ", value).strip()

raw_clues["category"] = raw_clues["category"].map(clean_text)
raw_clues["clue_text"] = raw_clues["clue_text"].map(clean_text)
raw_clues["round"] = raw_clues["round"].astype(str).str.strip().str.upper()
raw_clues["clue_value"] = pd.to_numeric(raw_clues["clue_value"], errors="coerce")

clues = raw_clues.drop_duplicates(
    subset=["game_id", "clue_source_record_id"], keep="first"
).copy()
clues = clues[(clues["category"] != "") & (clues["clue_text"] != "")].copy()

ordinary_rounds = {"J!", "DJ!", "1", "2"}
clues["is_ordinary"] = clues["round"].isin(ordinary_rounds) & clues["clue_value"].notna()
round_max = clues.loc[clues["is_ordinary"]].groupby(
    ["game_id", "round"], as_index=False
)["clue_value"].max().rename(columns={"clue_value": "round_max_value"})
clues = clues.merge(round_max, on=["game_id", "round"], how="left")
clues["relative_value"] = clues["clue_value"] / clues["round_max_value"]
clues["high_value_clue"] = (
    clues["is_ordinary"] & (clues["relative_value"] >= HIGH_VALUE_RATIO)
)

contestants["profile_text"] = contestants["occupation_text_for_model"].fillna("").map(clean_text)

assert len(contestants) == 11_646
assert contestants["game_id"].nunique() == 3_882
assert contestants.groupby("game_id").size().eq(3).all()
assert contestants.groupby("game_id")["is_winner"].sum().eq(1).all()
assert not clues.duplicated(["game_id", "clue_source_record_id"]).any()
assert contestants["profile_text"].ne("").all()

categories = clues[["game_id", "category"]].drop_duplicates().copy()
print(f"Distinct board clues: {len(clues):,}")
print(f"Distinct game-category pairs: {len(categories):,}")
print(f"High-value ordinary clues: {int(clues['high_value_clue'].sum()):,}")
print(f"Games without high-value clues: {clues.groupby('game_id')['high_value_clue'].sum().eq(0).sum():,}")

Distinct board clues: 225,388
Distinct game-category pairs: 50,200
High-value ordinary clues: 87,315
Games without high-value clues: 0


In [7]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    get_ipython().run_line_magic("pip", "install -q sentence-transformers scikit-learn")
    from sentence_transformers import SentenceTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

encoder = SentenceTransformer(EMBEDDING_MODEL_NAME)

unique_texts = pd.Index(
    pd.concat([
        contestants["profile_text"],
        clues["clue_text"],
        categories["category"],
    ], ignore_index=True).unique()
)
embeddings = encoder.encode(
    unique_texts.tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("float32")
text_to_embedding = {
    text: embeddings[index] for index, text in enumerate(unique_texts)
}

print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Unique texts embedded: {len(unique_texts):,}")
print(f"Embedding dimension: {embeddings.shape[1]}")

/opt/anaconda3/envs/my_jupyter_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 4046/4046 [04:39<00:00, 14.48it/s]


Embedding model: all-MiniLM-L6-v2
Unique texts embedded: 258,932
Embedding dimension: 384


In [8]:
feature_rows = []
for game_id, game_contestants in contestants.groupby("game_id", sort=False):
    game_clues = clues[clues["game_id"] == game_id]
    clue_texts = game_clues["clue_text"].tolist()
    category_texts = game_clues["category"].drop_duplicates().tolist()
    high_value_texts = game_clues.loc[
        game_clues["high_value_clue"], "clue_text"
    ].tolist()

    clue_vectors = np.vstack([text_to_embedding[text] for text in clue_texts])
    category_vectors = np.vstack([text_to_embedding[text] for text in category_texts])
    high_value_vectors = (
        np.vstack([text_to_embedding[text] for text in high_value_texts])
        if high_value_texts else None
    )

    for row in game_contestants.itertuples(index=False):
        profile_vector = text_to_embedding[row.profile_text]
        clue_similarities = clue_vectors @ profile_vector
        category_similarities = category_vectors @ profile_vector
        high_value_similarities = (
            high_value_vectors @ profile_vector
            if high_value_vectors is not None else np.array([np.nan])
        )
        feature_rows.append({
            "game_id": row.game_id,
            "contestant_id": row.contestant_id,
            "contestant_position": row.contestant_position,
            "is_winner": row.is_winner,
            "avg_similarity": float(clue_similarities.mean()),
            "max_similarity": float(clue_similarities.max()),
            "category_similarity": float(category_similarities.mean()),
            "high_value_similarity": float(np.nanmean(high_value_similarities)),
            "high_value_similarity_available": int(high_value_vectors is not None),
        })

features = pd.DataFrame(feature_rows)
base_feature_names = [
    "avg_similarity", "max_similarity", "category_similarity",
    "high_value_similarity",
]
for feature_name in base_feature_names:
    other_mean = features.groupby("game_id")[feature_name].transform("sum")
    other_mean = (other_mean - features[feature_name]) / 2.0
    features[f"{feature_name}_advantage"] = features[feature_name] - other_mean

advantage_feature_names = [f"{name}_advantage" for name in base_feature_names]
model_feature_names = base_feature_names + advantage_feature_names

assert len(features) == len(contestants)
assert features[model_feature_names].notna().all().all()
assert np.isfinite(features[model_feature_names].to_numpy()).all()
assert np.allclose(
    features.groupby("game_id")[advantage_feature_names].sum().to_numpy(),
    0.0,
    atol=1e-6,
)

print(f"Model rows: {len(features):,}")
display(features[model_feature_names].describe().T.round(4))

Model rows: 11,646


,count,mean,std,min,25%,50%,75%,max
avg_similarity,11646.0,0.0144,0.0170,-0.0531,0.0025,0.0133,0.0252,0.1081
max_similarity,11646.0,0.2292,0.0674,0.0589,0.1809,0.2212,0.2673,0.6198
category_similarity,11646.0,0.1358,0.0498,-0.0639,0.1017,0.1338,0.1691,0.3223
high_value_similarity,11646.0,0.0139,0.0204,-0.0606,-0.0003,0.0129,0.0269,0.1224
avg_similarity_advantage,11646.0,0.0000,0.0141,-0.0729,-0.0094,-0.0000,0.0092,0.0611
max_similarity_advantage,11646.0,0.0000,0.0739,-0.3305,-0.0482,-0.0035,0.0438,0.3942
category_similarity_advantage,11646.0,0.0000,0.0528,-0.2274,-0.0347,0.0008,0.0361,0.1943
high_value_similarity_advantage,11646.0,0.0000,0.0179,-0.0801,-0.0117,-0.0001,0.0116,0.0791


In [9]:
splitter = GroupShuffleSplit(
    n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
train_indices, test_indices = next(
    splitter.split(features, groups=features["game_id"])
)
train_game_ids = set(features.iloc[train_indices]["game_id"])
test_game_ids = set(features.iloc[test_indices]["game_id"])
assert train_game_ids.isdisjoint(test_game_ids)

features_by_position = features.sort_values(
    ["game_id", "contestant_position"]
).copy()
wide = features_by_position.pivot(
    index="game_id", columns="contestant_position", values=model_feature_names
)
wide.columns = [f"{feature}_position_{position}" for feature, position in wide.columns]
wide = wide.reset_index()

winner_positions = features.loc[features["is_winner"].eq(1)].set_index(
    "game_id"
)["contestant_position"]
wide["winner_position"] = wide["game_id"].map(winner_positions).astype(int)
wide["game_id"] = wide["game_id"].astype(features["game_id"].dtype)

wide_feature_names = [
    column for column in wide.columns
    if column not in {"game_id", "winner_position"}
]
X = wide[wide_feature_names]
y = wide["winner_position"]
groups = wide["game_id"]
train_mask = groups.isin(train_game_ids).to_numpy()
test_mask = groups.isin(test_game_ids).to_numpy()

model_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("logistic", LogisticRegression(
        solver="lbfgs",
        max_iter=2_000,
        random_state=RANDOM_STATE,
    )),
])

regularization_grid = {"logistic__C": np.logspace(-3, 3, 7)}
inner_cv = GroupKFold(n_splits=5)
search = GridSearchCV(
    model_pipeline,
    regularization_grid,
    scoring="neg_log_loss",
    cv=inner_cv,
    n_jobs=-1,
    refit=True,
)
search.fit(X.loc[train_mask], y.loc[train_mask], groups=groups.loc[train_mask])

heldout_probabilities = search.predict_proba(X.loc[test_mask])
heldout_predictions = search.predict(X.loc[test_mask])
class_order = search.best_estimator_.named_steps["logistic"].classes_

assert np.array_equal(class_order, np.array([1, 2, 3]))
assert np.allclose(heldout_probabilities.sum(axis=1), 1.0, atol=1e-8)

print(f"Training games: {train_mask.sum():,}")
print(f"Held-out games: {test_mask.sum():,}")
print(f"Best C: {search.best_params_['logistic__C']}")

Training games: 3,105
Held-out games: 777
Best C: 0.001


In [10]:
test_games = wide.loc[test_mask, ["game_id", "winner_position"]].reset_index(drop=True)
predictions = test_games.copy()
for index, position in enumerate(class_order):
    predictions[f"probability_position_{position}"] = heldout_probabilities[:, index]
predictions["predicted_winner_position"] = heldout_predictions
predictions["probability_sum"] = predictions[
    [f"probability_position_{position}" for position in class_order]
].sum(axis=1)

uniform_probabilities = np.full_like(heldout_probabilities, 1 / 3)
metrics = pd.DataFrame([
    {
        "model": "occupation_only_sentence_bert",
        "log_loss": log_loss(y.loc[test_mask], heldout_probabilities, labels=class_order),
        "accuracy": accuracy_score(y.loc[test_mask], heldout_predictions),
        "games": int(test_mask.sum()),
        "best_C": float(search.best_params_["logistic__C"]),
    },
    {
        "model": "uniform_reference",
        "log_loss": log_loss(y.loc[test_mask], uniform_probabilities, labels=class_order),
        "accuracy": np.nan,
        "games": int(test_mask.sum()),
        "best_C": np.nan,
    },
])

split_ids = pd.DataFrame({
    "game_id": wide["game_id"],
    "split": np.where(train_mask, "train", "test"),
})
feature_summary = features[model_feature_names].describe().T.reset_index()
feature_summary = feature_summary.rename(columns={"index": "feature"})

predictions.to_csv(OUTPUT_DIR / f"{ARTIFACT_PREFIX}_test_predictions.csv", index=False)
metrics.to_csv(OUTPUT_DIR / f"{ARTIFACT_PREFIX}_metrics.csv", index=False)
features.to_csv(OUTPUT_DIR / f"{ARTIFACT_PREFIX}_features.csv", index=False)
feature_summary.to_csv(OUTPUT_DIR / f"{ARTIFACT_PREFIX}_feature_summary.csv", index=False)
split_ids.to_csv(OUTPUT_DIR / f"{ARTIFACT_PREFIX}_game_splits.csv", index=False)

assert predictions["game_id"].nunique() == 777
assert np.allclose(predictions["probability_sum"], 1.0, atol=1e-8)
assert not set(split_ids.loc[split_ids["split"] == "train", "game_id"]).intersection(
    set(split_ids.loc[split_ids["split"] == "test", "game_id"])
)

display(metrics.round(4))
print(f"Saved occupation-only model artifacts to {OUTPUT_DIR}")

,model,log_loss,accuracy,games,best_C
0,occupation_only_sentence_bert,1.0968,0.3784,777,0.001
1,uniform_reference,1.0986,NaN,777,NaN


Saved occupation-only model artifacts to /Users/Bhargav/Code/DS4002-SLMB/Project_1/Output
